In [1]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# Step 1: Get app metadata (rating, installs, description...) for CBE, BOA, Dashen banks

# Map clean display names to their respective Play Store App IDs

banks = {
    "Commercial Bank of Ethiopia": "com.combanketh.mobilebanking",
    "Bank of Abyssinia": "com.boa.boaMobileBanking",
    "Dashen Bank": "com.dashen.dashensuperapp"
}

# Iteratively fetch and display app metadata
for bank_name, app_id in banks.items():
    try:
    # Fetch data directly inside the loop using the current app_id
        app_info = app(app_id, lang='en', country='et')     # Language: English, Country: Ethiopia
    
        print("=" * 50)
        print(f"{bank_name} App Info")
        print("=" * 50)
        print(f"App Title    : {app_info['title']}")
        print(f"Current Score: {app_info['score']}")
        print(f"Total Ratings: {app_info['ratings']:,}")
        print(f"Total Reviews: {app_info['reviews']:,}")
        print(f"Installs     : {app_info['installs']}\n")

    except Exception as e:
        # If an ID fails (like a 404), catch it here and keep going
        print("=" * 50)
        print(f"⚠️ Error loading {bank_name}")
        print("=" * 50)
        print(f"Could not retrieve App ID: '{app_id}'")
        print(f"Details: {e}\n")

Commercial Bank of Ethiopia App Info
App Title    : Commercial Bank of Ethiopia
Current Score: 4.2835107
Total Ratings: 48,537
Total Reviews: 9,329
Installs     : 10,000,000+

Bank of Abyssinia App Info
App Title    : BoA Mobile
Current Score: 4.3978496
Total Ratings: 9,276
Total Reviews: 1,465
Installs     : 1,000,000+

Dashen Bank App Info
App Title    : Dashen Bank
Current Score: 4.258333
Total Ratings: 5,686
Total Reviews: 1,026
Installs     : 1,000,000+



In [15]:
# Step 2: Scrape reviews

# Dictionary to map clean display names to their App IDs
banks = {
    "Commercial Bank of Ethiopia": "com.combanketh.mobilebanking",
    "Bank of Abyssinia": "com.boa.boaMobileBanking",
    "Dashen Bank": "com.dashen.dashensuperapp"
}

# Master dictionary to store the actual raw reviews lists for each bank
all_bank_reviews = {}

# 1. Iterate through each bank to scrape reviews
for bank_name, app_id in banks.items():  
    try:
        # Scrape reviews
        result, continuation_token = reviews(
            app_id,
            lang='en',
            country='et',
            sort=Sort.NEWEST,       # Most recent first
            count=1000,             # Target count
            filter_score_with=None  # All star ratings
        )
        
        # Store the list of reviews using the bank's name as the key
        all_bank_reviews[bank_name] = result
        print(f"✅ Success: Collected {len(result)} raw reviews for {bank_name}\n")
        
    except Exception as e:
        # **Safety Net:** If one bank fails (e.g., a temporary network glitch or an ID change), it catches errors and keep the loop running
        print(f"❌ Error scraping {bank_name} due to error: {e}\n")

# --- FINAL VALIDATION CHECK ---
print("=" * 50)
print("FINAL COLLECTION CHECK SUMMARY")
print("=" * 50)
for bank, status in all_bank_reviews.items():
    print(f"{bank:<30} : {len(status)} Raw Reviews")
print("=" * 50)

# 2. Inspect a single raw review from each bank
print("=" * 60)
print("INSPECTING A SAMPLE REVIEW FOR EACH BANK")
print("=" * 60)

# Iterate through all banks in the collected data
for bank_name, reviews_list in all_bank_reviews.items():
    print(f"\nTarget Bank: {bank_name}")
    print("-" * 50)
    
    if reviews_list and len(reviews_list) > 0:
        # Grab the very first review dictionary from the current bank's list
        sample_review = reviews_list[0]
        
        # Display all the available keys safely
        print(f"Keys available in this review: {list(sample_review.keys())}\n")
        print("First raw review data details:")
        
        for key, value in sample_review.items():
            print(f" {key:<20}: {value}")
            
    else:
        print(f"No sample data available for {bank_name}. Check your collection step.")
    print("-" * 50)

✅ Success: Collected 1000 raw reviews for Commercial Bank of Ethiopia

✅ Success: Collected 1000 raw reviews for Bank of Abyssinia

✅ Success: Collected 1000 raw reviews for Dashen Bank

FINAL COLLECTION CHECK SUMMARY
Commercial Bank of Ethiopia    : 1000 Raw Reviews
Bank of Abyssinia              : 1000 Raw Reviews
Dashen Bank                    : 1000 Raw Reviews
INSPECTING A SAMPLE REVIEW FOR EACH BANK

Target Bank: Commercial Bank of Ethiopia
--------------------------------------------------
Keys available in this review: ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review data details:
 reviewId            : 72fd0808-c8f7-435a-8337-0308012763da
 userName            : ssmson tadesse
 userImage           : https://play-lh.googleusercontent.com/a-/ALV-UjWmgH4WcTqdc1CvIyRiwZKMPeyIf-t8XoyWuBmMxfDFa1r4xkUC
 content             : nays
 score               : 5
 thumbsUpCount 

In [18]:
# Step 3: Extract only the columns needed from all banks
combined_raw_data = []

for bank_name, reviews_list in all_bank_reviews.items():
    print(f"Processing and extracting columns for: {bank_name}...")
    
    for r in reviews_list:
        combined_raw_data.append({
            'review_id': r.get('reviewId', ''),
            'review'   : r.get('content', ''),
            'rating'   : r.get('score', None),
            'date'     : r.get('at', None),
            'bank'     : bank_name,          # Dynamically tags the correct bank name
            'source'   : 'Google Play'
        })

# Build a single master DataFrame
df_raw = pd.DataFrame(combined_raw_data)

print("\n" + "=" * 50)
print(f"Extraction Complete!")
print(f"Final Combined DataFrame Shape: {df_raw.shape}")
print("=" * 50)

# Display a breakdown of reviews collected per bank
print("\nReviews per bank in DataFrame:")
print(df_raw['bank'].value_counts())

# Randomly select and display 5 reviews in a table format
df_raw.sample(5)

Processing and extracting columns for: Commercial Bank of Ethiopia...
Processing and extracting columns for: Bank of Abyssinia...
Processing and extracting columns for: Dashen Bank...

Extraction Complete!
Final Combined DataFrame Shape: (3000, 6)

Reviews per bank in DataFrame:
bank
Commercial Bank of Ethiopia    1000
Bank of Abyssinia              1000
Dashen Bank                    1000
Name: count, dtype: int64


,review_id,review,rating,date,bank,source
2267,853d49b8-e54a-459e-8011-6ad199abb578,The apk need time to open the passward page im...,1,2025-12-01 08:48:25,Dashen Bank,Google Play
1361,a7eb5cbd-790a-4537-8278-ccaef158654b,1) Crashes repeatedly 2) Takes Century to Boot...,1,2025-08-02 16:36:41,Bank of Abyssinia,Google Play
1739,ff078b88-0b15-49b1-8222-9d9d6125bb1b,Ok,5,2024-08-06 15:33:00,Bank of Abyssinia,Google Play
108,85564924-a1f4-4a11-933e-5c3c0ede7875,It's not working after the update it doesn't o...,1,2026-04-30 10:29:38,Commercial Bank of Ethiopia,Google Play
928,992a7925-57f1-4bed-be83-5015ac2e15c4,good service,5,2026-01-06 19:51:02,Commercial Bank of Ethiopia,Google Play


In [19]:
# A) Exploring the merged raw data

# Basic shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)


# Rating distribution — what do users think?
print("\nRating distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")


# What does the date column look like right now?
print("\nSample date values (raw):")
print(df_raw['date'].sample(5).to_string())   #Randomly select date to notice differences

print(f"\nDate dtype: {df_raw['date'].dtype}")

Total reviews collected: 3000

Column dtypes:
review_id            object
review               object
rating                int64
date         datetime64[ns]
bank                 object
source               object
dtype: object

Rating distribution:
  5 stars: 1852  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  4 stars:  199  ███████████████████████████████████████
  3 stars:  161  ████████████████████████████████
  2 stars:  119  ███████████████████████
  1 stars:  669  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

Sample date values (raw):
581    2026-02-20 23:31:44
634    20

In [20]:
# B) Data Quality Audit

print("=" * 50)
print("DATA QUALITY AUDIT")
print("=" * 50)

# --- Problem 1: Missing Values ---
print("\nProblem 1: Missing Values")
print("-" * 30)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

for col in df_raw.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

# --- Problem 2: Duplicate Reviews ---
print("\nProblem 2: Duplicates")
print("-" * 30)

# Exact duplicates on review text
exact_dupes = df_raw.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes}")

# Duplicate review IDs
id_dupes = df_raw.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes}")

# Empty reviews (also a form of bad data)
empty_reviews = (df_raw['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews}")

# --- Problem 3: Date Format ---
print("\nProblem 3: Date Format")
print("-" * 30)
print(f"  Current dtype: {df_raw['date'].dtype}")
print(f"  Sample values: {df_raw['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")


# --- Problem 4: Language Check ---
import pandas as pd
from langdetect import detect, DetectorFactory  # To detect language differences
from collections import Counter
# Ensure deterministic language identification results
DetectorFactory.seed = 0

def safe_detect_lang(text):
    if not isinstance(text, str) or text.strip() == "":
        return "unknown/empty"
    try:
        # Returns standard codes like 'en', 'am', etc.
        return detect(text.strip())
    except:
        return "detection_error"

# 1. Apply language profiling across the dataframe copy
print("🔍 Analyzing dataset languages... (This may take a moment)")
df_raw['detected_lang'] = df_raw['review'].apply(safe_detect_lang)

# 2. Iterate through each bank to calculate the breakdown
print("\n" + "=" * 60)
print(" 🌍 LANGUAGE PROFILE BREAKDOWN BY BANK")
print("=" * 60)

for bank in df_raw['bank'].unique():
    df_bank = df_raw[df_raw['bank'] == bank]
    total_reviews = len(df_bank)
    
    print(f"\n🏢 {bank.upper()}")
    print(f"Total reviews checked: {total_reviews}")
    
    # Count occurrences of each language
    lang_counts = Counter(df_bank['detected_lang'])
    
    # Display languages making up more than 1% of the bank's data
    for lang, count in lang_counts.most_common(5):
        percentage = (count / total_reviews) * 100
        
        # Friendly translations for common outcomes in Ethiopian app stores
        lang_label = "English" if lang == 'en' else ("Amharic" if lang == 'am' else f"Other Code ('{lang}')")
        
        print(f"  - {lang_label:<15}: {count:>4} records ({percentage:>5.1f}%)")
    print("-" * 45)

DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
  review_id      : OK
  review         : OK
  rating         : OK
  date           : OK
  bank           : OK
  source         : OK

Problem 2: Duplicates
------------------------------
  Exact duplicate reviews : 705
  Duplicate review IDs    : 0
  Empty review texts      : 0

Problem 3: Date Format
------------------------------
  Current dtype: datetime64[ns]
  Sample values: 2026-05-17 18:46:14
  Target format: YYYY-MM-DD (string or date object)
🔍 Analyzing dataset languages... (This may take a moment)

 🌍 LANGUAGE PROFILE BREAKDOWN BY BANK

🏢 COMMERCIAL BANK OF ETHIOPIA
Total reviews checked: 1000
  - English        :  498 records ( 49.8%)
  - Other Code ('so'):  139 records ( 13.9%)
  - Other Code ('af'):   64 records (  6.4%)
  - Other Code ('detection_error'):   50 records (  5.0%)
  - Other Code ('pl'):   35 records (  3.5%)
---------------------------------------------

🏢 BANK OF ABYSSINIA
Total revi

In [21]:
# C) Cleaning Strategy

# Step 0:- Work on a copy so raw data stays untouched
df = df_raw.copy()

print(f"Starting with: {len(df)} reviews")
print("-" * 50)

# Step 1:- Handle Missing Values

before = len(df)

# Drop rows missing the critical columns
critical_cols = ['review', 'rating']
df = df.dropna(subset=critical_cols)

removed = before - len(df)
print(f"\nRemoved {removed} rows with missing critical data")
print(f"Remaining: {len(df)} reviews")

# Step 2:- Remove Duplicates

before = len(df)

df = df.drop_duplicates(subset=['review_id'], keep='first')

removed = before - len(df)
print(f"\nRemoved {removed} duplicate reviews")
print(f"Remaining: {len(df)} reviews")

# Step 3:- Normalize Dates

print("\nBefore normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

# Convert to pandas datetime, then format as YYYY-MM-DD string
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

print("\nAfter normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

# Step 4:- Remove reviews with other languages except for English

# 1. Filter out only the verified English ('en') reviews from df_raw
df_english_all = df[df['detected_lang'] == 'en']

# 2. Slices exactly the first 400 English records per bank
balanced_chunks = []
target_sample_size = 400

for bank in df_english_all['bank'].unique():
    # Isolate English reviews for this specific bank
    df_bank_en = df_english_all[df_english_all['bank'] == bank]
    
    # Grab the first 400 sequential rows using .head()
    df_bank_sliced = df_bank_en.head(target_sample_size)
    
    print(f"✂️ {bank.upper()}: Captured the first {len(df_bank_sliced)} English reviews.")
    balanced_chunks.append(df_bank_sliced)

# 3. Combine them into the final working DataFrame
df_clean = pd.concat(balanced_chunks, ignore_index=True)

print("=" * 60)
print(f"✅ Success! 'df_clean' now contains a perfectly balanced English dataset.\n")
print(df_clean['bank'].value_counts())
print("=" * 60)

# Step 5:- Clean Review Text

def clean_text(text):
    """Standardize review text: collapse whitespace, strip edges."""
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines
    text = text.strip()               # remove leading/trailing whitespace
    return text

# Show before/after on a sample review
sample_raw = "  Great   app!\n\nVery useful.  "
print(f"\nBefore: {repr(sample_raw)}")
print(f"After : {repr(clean_text(sample_raw))}")

# Apply to the full column
df_clean['review'] = df_clean['review'].apply(clean_text)

# Remove any reviews that became empty after cleaning
before_strip= len(df_clean)
df_clean = df_clean[df_clean['review'].str.len() > 0]
removed_blank = before_strip - len(df_clean)
print(f"\nRemoved {removed_blank} reviews that were empty after cleaning")

# Step 6:- Validate Ratings

# Check for out-of-range ratings
invalid_ratings = df_clean[(df_clean['rating'] < 1) | (df_clean['rating'] > 5)]
print(f"\nInvalid ratings (outside 1–5): {len(invalid_ratings)}")

# Remove them
df_clean = df_clean[(df_clean['rating'] >= 1) & (df_clean['rating'] <= 5)]

# Ensure rating is stored as integer
df_clean['rating'] = df_clean['rating'].astype(int)

print(f"Remaining: {len(df_clean)} reviews")
print(f"Rating dtype: {df_clean['rating'].dtype}")

Starting with: 3000 reviews
--------------------------------------------------

Removed 0 rows with missing critical data
Remaining: 3000 reviews

Removed 0 duplicate reviews
Remaining: 3000 reviews

Before normalization:
0   2026-05-17 18:46:14
1   2026-05-17 17:03:51
2   2026-05-17 16:12:24
dtype: datetime64[ns]

After normalization:
0    2026-05-17
1    2026-05-17
2    2026-05-17
dtype: object

Date range: 2024-05-03 to 2026-05-17
✂️ COMMERCIAL BANK OF ETHIOPIA: Captured the first 400 English reviews.
✂️ BANK OF ABYSSINIA: Captured the first 400 English reviews.
✂️ DASHEN BANK: Captured the first 400 English reviews.
✅ Success! 'df_clean' now contains a perfectly balanced English dataset.

bank
Commercial Bank of Ethiopia    400
Bank of Abyssinia              400
Dashen Bank                    400
Name: count, dtype: int64

Before: '  Great   app!\n\nVery useful.  '
After : 'Great app! Very useful.'

Removed 0 reviews that were empty after cleaning

Invalid ratings (outside 1–5): 0


In [22]:
# Step 6:- Final output

# Select only the 5 required columns in the right order
df_processed = df_clean[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_processed = df_processed.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_processed.shape}")
df_processed.head(10)

Final dataset shape: (1200, 5)


,review,rating,date,bank,source
0,very very good 👍 thanks commercial bank of eth...,5,2026-05-17,Commercial Bank of Ethiopia,Google Play
1,sometimes The App Is not goibg through,3,2026-05-17,Bank of Abyssinia,Google Play
2,The Bank You can Always Rely on!,5,2026-05-17,Commercial Bank of Ethiopia,Google Play
3,"It is a very cool application, but it requires...",4,2026-05-17,Dashen Bank,Google Play
4,Please make the CBE Noor toggle to be optional...,2,2026-05-17,Commercial Bank of Ethiopia,Google Play
5,this app very full,5,2026-05-16,Commercial Bank of Ethiopia,Google Play
6,It stopped working on its own. When you check ...,1,2026-05-16,Commercial Bank of Ethiopia,Google Play
7,Misguiding - Claimed anyone can convert ETB to...,1,2026-05-16,Dashen Bank,Google Play
8,"The worst app, also bank am begging for my own...",1,2026-05-16,Bank of Abyssinia,Google Play
9,The most backward and unstable financial app i...,1,2026-05-16,Commercial Bank of Ethiopia,Google Play


In [24]:
# Save to CSV
import os

target_directory = r'C:\Users\nemsa\KAIM Challenge Project\Fintech-Review-Analytics\data\processed'
os.makedirs(target_directory, exist_ok=True)

output_path = os.path.join(target_directory, 'combined_bank_reviews_cleaned.csv')
df_processed.to_csv(output_path, index=False)

print(f"✅ Master dataset successfully saved to:\n{output_path}")

✅ Master dataset successfully saved to:
C:\Users\nemsa\KAIM Challenge Project\Fintech-Review-Analytics\data\processed\combined_bank_reviews_cleaned.csv


In [25]:
# Step 7: Preprocessing Report

print("=" * 55)
print("  PREPROCESSING REPORT — CBE, BOA, Dashen Banks Reviews")
print("=" * 55)

original_count = len(df_raw)
final_count    = len(df_processed)
removed_total  = original_count - final_count
retention_rate = (final_count / original_count * 100)

print(f"\n  Raw reviews collected  : {original_count:>6}")
print(f"  Reviews after cleaning : {final_count:>6}")
print(f"  Reviews removed        : {removed_total:>6}")
print(f"  Data retention rate    : {retention_rate:>5.1f}%")

quality = "EXCELLENT" if retention_rate >= 95 else ("GOOD" if retention_rate >= 90 else "NEEDS ATTENTION")
print(f"  Data quality           : {quality}")

print(f"\n  Date range : {df_processed['date'].min()}  to  {df_processed['date'].max()}")

print("\n  Rating distribution:")
for rating in sorted(df_processed['rating'].unique(), reverse=True):
    count = (df_processed['rating'] == rating).sum()
    pct   = count / final_count * 100
    bar   = '█' * (count // 5)
    print(f"    {rating} stars : {count:>4} ({pct:4.1f}%)  {bar}")

print("\n  Text length stats:")
lengths = df_processed['review'].str.len()
print(f"    Min    : {lengths.min()} characters")
print(f"    Median : {lengths.median():.0f} characters")
print(f"    Max    : {lengths.max()} characters")

print("\n  Columns in final CSV:")
for col in df_processed.columns:
    print(f"    - {col}")

print("\n" + "=" * 55)

  PREPROCESSING REPORT — CBE, BOA, Dashen Banks Reviews

  Raw reviews collected  :   3000
  Reviews after cleaning :   1200
  Reviews removed        :   1800
  Data retention rate    :  40.0%
  Data quality           : NEEDS ATTENTION

  Date range : 2024-08-01  to  2026-05-17

  Rating distribution:
    5 stars :  573 (47.8%)  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████
    4 stars :   86 ( 7.2%)  █████████████████
    3 stars :   83 ( 6.9%)  ████████████████
    2 stars :   69 ( 5.8%)  █████████████
    1 stars :  389 (32.4%)  █████████████████████████████████████████████████████████████████████████████

  Text length stats:
    Min    : 3 characters
    Median : 43 characters
    Max    : 500 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source



In [26]:
import pandas as pd

# List of banks to process based on your combined dataset
banks_to_report = ["Commercial Bank of Ethiopia", "Bank of Abyssinia", "Dashen Bank"]

for bank in banks_to_report:
    # Filter dataframes for the current bank
    df_raw_bank = df_raw[df_raw['bank'] == bank]
    df_processed_bank = df_processed[df_processed['bank'] == bank]
    
    # Skip if no data exists for this bank to prevent division-by-zero errors
    if len(df_raw_bank) == 0:
        print(f"\n⚠️ No data found for {bank}. Skipping...")
        continue

    print("=" * 55)
    print(f"   PREPROCESSING REPORT — {bank}")
    print("=" * 55)

    original_count = len(df_raw_bank)
    final_count    = len(df_processed_bank)
    removed_total  = original_count - final_count
    retention_rate = (final_count / original_count * 100)

    print(f"\n  Raw reviews collected  : {original_count:>6}")
    print(f"  Reviews after cleaning : {final_count:>6}")
    print(f"  Reviews removed        : {removed_total:>6}")
    print(f"  Data retention rate    : {retention_rate:>5.1f}%")

    quality = "EXCELLENT" if retention_rate >= 95 else ("GOOD" if retention_rate >= 90 else "NEEDS ATTENTION")
    print(f"  Data quality           : {quality}")

    # Handle missing dates gracefully if all records were cleared
    if final_count > 0:
        print(f"\n  Date range : {df_processed_bank['date'].min()}  to  {df_processed_bank['date'].max()}")

        print("\n  Rating distribution:")
        for rating in sorted(df_processed_bank['rating'].unique(), reverse=True):
            count = (df_processed_bank['rating'] == rating).sum()
            pct   = count / final_count * 100
            bar   = '█' * (count // 5)  # Visual scale helper
            print(f"    {rating} stars : {count:>4} ({pct:4.1f}%)  {bar}")

        print("\n  Text length stats:")
        lengths = df_processed_bank['review'].str.len()
        print(f"    Min    : {lengths.min()} characters")
        print(f"    Median : {lengths.median():.0f} characters")
        print(f"    Max    : {lengths.max()} characters")
    else:
        print("\n  ⚠️ No records remaining after data cleaning step.")

    print("\n  Columns in final CSV:")
    for col in df_processed.columns:
        print(f"    - {col}")

    print("\n" + "=" * 55 + "\n")

   PREPROCESSING REPORT — Commercial Bank of Ethiopia

  Raw reviews collected  :   1000
  Reviews after cleaning :    400
  Reviews removed        :    600
  Data retention rate    :  40.0%
  Data quality           : NEEDS ATTENTION

  Date range : 2026-01-30  to  2026-05-17

  Rating distribution:
    5 stars :  202 (50.5%)  ████████████████████████████████████████
    4 stars :   37 ( 9.2%)  ███████
    3 stars :   33 ( 8.2%)  ██████
    2 stars :   28 ( 7.0%)  █████
    1 stars :  100 (25.0%)  ████████████████████

  Text length stats:
    Min    : 3 characters
    Median : 38 characters
    Max    : 500 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source


   PREPROCESSING REPORT — Bank of Abyssinia

  Raw reviews collected  :   1000
  Reviews after cleaning :    400
  Reviews removed        :    600
  Data retention rate    :  40.0%
  Data quality           : NEEDS ATTENTION

  Date range : 2024-08-01  to  2026-05-17

  Rating distribu